In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [2]:
import torch
import torch.nn as nn

from configs.config import config
print(config)
print(config.embed_dim)

GPTConfig(vocab_size=32000, max_seq_len=256, embed_dim=256, num_heads=8, num_layers=6, ffn_dim=1024, dropout=0.1, batch_size=16, learning_rate=0.0003, weight_decay=0.01, epochs=10, tokenizer_path=WindowsPath('C:/Users/FH/Documents/PersionLLM/PersianGPT/data/tokenizer/persian.model'), train_file=WindowsPath('C:/Users/FH/Documents/PersionLLM/PersianGPT/data/processed/clean_corpus.txt'), checkpoint_dir=WindowsPath('C:/Users/FH/Documents/PersionLLM/PersianGPT/checkpoints'), device='cuda')
256


In [3]:
print("CUDA Available :", torch.cuda.is_available())

device = torch.device(config.device if torch.cuda.is_available() else "cpu")

print("Using Device :", device)

CUDA Available : True
Using Device : cuda


In [5]:
class PersianGPT(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.config = config

        # Token Embedding
        self.token_embedding = nn.Embedding(
            config.vocab_size,
            config.embed_dim,
        )

        # Positional Embedding
        self.position_embedding = nn.Embedding(
            config.max_seq_len,
            config.embed_dim,
        )

        self.dropout = nn.Dropout(config.dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.embed_dim,
            nhead=config.num_heads,
            dim_feedforward=config.ffn_dim,
            dropout=config.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=config.num_layers,
        )

        self.ln_f = nn.LayerNorm(config.embed_dim)

        self.lm_head = nn.Linear(
            config.embed_dim,
            config.vocab_size,
            bias=False,
        )
        
        # Weight Tying
        self.lm_head.weight = self.token_embedding.weight
    

    def forward(self, input_ids):

        batch_size, seq_len = input_ids.shape

        device = input_ids.device

        positions = torch.arange(
            seq_len,
            device=device
        ).unsqueeze(0)

        token_embeddings = self.token_embedding(input_ids)

        position_embeddings = self.position_embedding(positions)

        x = token_embeddings + position_embeddings

        x = self.dropout(x)

        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=device),
            diagonal=1,
        ).bool()

        x = self.transformer(
            x,
            mask=causal_mask,
        )

        x = self.ln_f(x)

        logits = self.lm_head(x)

        return logits

In [6]:
model = PersianGPT(config).to(device)

print(model)

c:\Users\FH\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


PersianGPT(
  (token_embedding): Embedding(32000, 256)
  (position_embedding): Embedding(256, 256)
  (dropout): Dropout(p=0.1, inplace=False)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=1024, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_features=256, out_features=32000, bias=False)
)


In [7]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

Total Parameters     : 12,996,608
Trainable Parameters : 12,996,608


In [8]:
dummy_input = torch.randint(
    low=0,
    high=config.vocab_size,
    size=(2, config.max_seq_len),
    device=device,
)

print(dummy_input.shape)

torch.Size([2, 256])


In [9]:
with torch.no_grad():
    logits = model(dummy_input)

print(logits.shape)

torch.Size([2, 256, 32000])


In [10]:
print("Model Device :", next(model.parameters()).device)
print("Input Device :", dummy_input.device)
print("Output Device:", logits.device)

Model Device : cuda:0
Input Device : cuda:0
Output Device: cuda:0
